# Customer Segmentation with K-Means

**Type:** Original supplementary practice  
**Data:** Synthetic customer behaviour features

This exercise demonstrates scaling, selecting a cluster count with silhouette scores, interpreting cluster centres and projecting clusters into two dimensions. Clusters are mathematical groupings; they are not automatically meaningful customer personas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

X, _ = make_blobs(
    n_samples=1000, centers=4, n_features=4,
    cluster_std=[1.0,1.5,0.8,1.2], random_state=42
)
features = pd.DataFrame(X, columns=[
    "purchase_frequency","average_spend","product_breadth","discount_sensitivity"
])
features["purchase_frequency"] = features["purchase_frequency"]*2 + 12
features["average_spend"] = features["average_spend"]*140 + 900
features["product_breadth"] = np.maximum(1, features["product_breadth"]+5)
features["discount_sensitivity"] = 50 + 12*features["discount_sensitivity"]
display(features.describe().round(2))

## 1. Scale the features

K-Means uses distance calculations. Features with large numeric scales can dominate unless scaling is appropriate.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

## 2. Compare candidate cluster counts

In [ ]:
silhouette = {}
for k in range(2,8):
    labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X_scaled)
    silhouette[k] = silhouette_score(X_scaled, labels)

display(pd.Series(silhouette, name="silhouette_score").to_frame())
best_k = max(silhouette, key=silhouette.get)
print("Highest silhouette candidate:", best_k)

## 3. Fit and describe clusters

In [ ]:
kmeans = KMeans(n_clusters=best_k, n_init=30, random_state=42)
features["cluster"] = kmeans.fit_predict(X_scaled)
profile = features.groupby("cluster").agg(
    customer_count=("cluster","size"),
    purchase_frequency=("purchase_frequency","mean"),
    average_spend=("average_spend","mean"),
    product_breadth=("product_breadth","mean"),
    discount_sensitivity=("discount_sensitivity","mean")
)
display(profile.round(2))

## 4. Visualise with PCA

PCA compresses the four scaled features into two components for plotting. The plot can hide separations visible in the original feature space.

In [ ]:
coords = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
fig, ax = plt.subplots(figsize=(8,5))
scatter = ax.scatter(coords[:,0], coords[:,1], c=features["cluster"], alpha=.65, s=18)
ax.set(title="K-Means clusters projected with PCA", xlabel="PC1", ylabel="PC2")
plt.tight_layout()
plt.show()

## Limitations and next steps

Check stability across random seeds and samples; inspect whether clusters are useful for a real decision; compare with hierarchical clustering or DBSCAN; and validate profiles with domain experts. The generated clusters have no real customer identity or behavioural meaning.